# Transciptions for talks. 
- Download MP3 as talk titles 
- Use WisperAI to process MP3 attaching to name 
- Combine data in a dataframe. 
- Perform entity resolution on list of talks? 
    - Some talks may not have been recorded
    - Talk names may differ 



## Get Audio
note must have ffmpeg instaled 

In [2]:
import os
import yt_dlp
import pandas as pd

# EMF 2026 audio collection
PLAYLIST_URL = "https://media.ccc.de/c/emf2026/audio"

# Destination folder
DOWNLOAD_FOLDER = os.path.normpath(
    os.path.join("..", "Raw_data", "Recordings")
)


In [3]:
os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)

# Download configuration
options = {
    "format": "bestaudio/best",
    "outtmpl": os.path.join(
        DOWNLOAD_FOLDER,
        "%(playlist_index)03d - %(title)s.%(ext)s"
    ),

    # Download entire playlist
    "yes_playlist": True,

    # Avoid downloading files already present
    "ignoreerrors": True,

    # Show progress
    "progress": True,
}

print("EMF 2026 Audio Downloader")
print("-------------------------")
print(f"Destination: {DOWNLOAD_FOLDER}")
print("Starting download...\n")

with yt_dlp.YoutubeDL(options) as downloader:
    downloader.download([PLAYLIST_URL])

print("\nDownload process finished.")
print(f"Audio files saved in: {DOWNLOAD_FOLDER}")

EMF 2026 Audio Downloader
-------------------------
Destination: ..\Raw_data\Recordings
Starting download...

[media.ccc.de:lists] Extracting URL: https://media.ccc.de/c/emf2026/audio
[media.ccc.de:lists] emf2026: Downloading JSON metadata
[download] Downloading playlist: Electromagnetic Field 2026
[media.ccc.de:lists] Playlist Electromagnetic Field 2026: Downloading 130 items of 130
[download] Downloading item 1 of 130
[media.ccc.de] Extracting URL: https://media.ccc.de/v/emf2026-240-1-men-in-tech-yesallmen
[media.ccc.de] emf2026-240-1-men-in-tech-yesallmen: Downloading webpage
[media.ccc.de] 17954: Downloading JSON metadata
[info] 17954: Downloading 1 format(s): eng-opus
[download] ..\Raw_data\Recordings\001 - Men in tech #YesAllMen.opus has already been downloaded
[download] 100% of   17.63MiB
[download] Downloading item 2 of 130
[media.ccc.de] Extracting URL: https://media.ccc.de/v/emf2026-290-1-endocrine-biohacking
[media.ccc.de] emf2026-290-1-endocrine-biohacking: Downloading web

In [4]:
import re

schedule = pd.read_csv("../../EMF_Camp/Raw_data/schedule.csv")

# Get filenames from directory
audio_files = [
    filename
    for filename in os.listdir(DOWNLOAD_FOLDER)
    if filename.lower().endswith((".mp3", ".opus"))
]
audio_df = pd.DataFrame({"title": audio_files})
audio_df["path"] = audio_df["title"].map(
    lambda filename: os.path.abspath(os.path.join(DOWNLOAD_FOLDER, filename)) # Get absolute path of each audio file for later easy playback
)

def extract_title(filename, *, safe_filename=False):
    """Extract a recording title; optionally make it safe for use as a filename."""
    name = os.path.basename(os.fspath(filename))
    name = re.sub(r"\.(?:opus|mp3)$", "", name, flags=re.IGNORECASE) # Remove file extension
    name = re.sub(r"^\d+\s*-\s*", "", name).strip() # Remove leading numbers and dashes, and strip whitespace

    if safe_filename:
        name = re.sub(r'[<>:"/\\|?*]', "_", name).strip(" .") # Replace invalid filename characters with underscores and strip leading/trailing spaces and dots

    return name

audio_df["title"] = audio_df["title"].apply(extract_title)

audio_df

,title,path
0,Men in tech #YesAllMen,c:\Users\Graem\Desktop\Github Repos\Data Engin...
1,"Endocrine Biohacking, 2 years on (Everything i...",c:\Users\Graem\Desktop\Github Repos\Data Engin...
2,PowerPoint Karaoke： After Dark Edition,c:\Users\Graem\Desktop\Github Repos\Data Engin...
3,It’s the End of RSA as We Know It (And I Feel ...,c:\Users\Graem\Desktop\Github Repos\Data Engin...
4,EMF Variety Night,c:\Users\Graem\Desktop\Github Repos\Data Engin...
...,...,...
125,Hacking APRS to warm my camper van whilst I’m ...,c:\Users\Graem\Desktop\Github Repos\Data Engin...
126,Boiling A Frog： Science & Ethics as TTRPG,c:\Users\Graem\Desktop\Github Repos\Data Engin...
127,Rotundest Robin： Building a Smart Bird Feeder ...,c:\Users\Graem\Desktop\Github Repos\Data Engin...
128,Opening Ceremony,c:\Users\Graem\Desktop\Github Repos\Data Engin...


## Disect a bundled Package of FFmpeg to reduce user setup burden
docker eat your heartout.

In [14]:
import os
import shutil
import tempfile
import subprocess
from pathlib import Path
import imageio_ffmpeg
import whisper

# Locate the bundled FFmpeg executable
ffmpeg_source = Path(imageio_ffmpeg.get_ffmpeg_exe())
# Create a temporary folder containing an executable named ffmpeg.exe
ffmpeg_folder = Path(tempfile.gettempdir()) / "emf_camp_ffmpeg"
ffmpeg_folder.mkdir(exist_ok=True)
ffmpeg_alias = ffmpeg_folder / "ffmpeg.exe"
shutil.copy2(ffmpeg_source, ffmpeg_alias)
# Add the folder to PATH so Whisper can find "ffmpeg"
os.environ["PATH"] = (
    str(ffmpeg_folder) + os.pathsep + os.environ.get("PATH", "")
)

# Test FFmpeg
subprocess.run(
    ["ffmpeg", "-version"],
    check=True,
    capture_output=True
)
print("FFmpeg configured successfully!")
# Test Whisper audio loading
audio = whisper.load_audio(str(audio_files[0]))
print("Audio loaded successfully!")
print("Audio shape:", audio.shape)

FFmpeg configured successfully!
Audio loaded successfully!
Audio shape: (28781909,)


In [16]:
import whisper
import whisper.audio
import imageio_ffmpeg
import time
from pathlib import Path

# Tell Whisper to use the FFmpeg executable bundled with imageio-ffmpeg to reduce setup issues 
whisper.audio.FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
print("FFmpeg version:", imageio_ffmpeg.get_ffmpeg_version())

# Define folders
BASE_FOLDER = Path(DOWNLOAD_FOLDER)
TRANSCRIPT_FOLDER = BASE_FOLDER / "transcript"
TRANSCRIPT_FOLDER.mkdir(parents=True, exist_ok=True)
# Get original audio filenames (retain filenames for locating audio)
audio_files = sorted([
    filename
    for filename in BASE_FOLDER.iterdir()
    if filename.is_file()
    and filename.suffix.lower() in (".mp3", ".opus")
])
total_files = len(audio_files)
print(f"\nFound {total_files} recordings.")

# Choose Whisper model
MODEL_NAME = "tiny"
print(f"Loading Whisper model: {MODEL_NAME}")
model = whisper.load_model(MODEL_NAME)

# Start Transcription Process
if total_files == 0:
    print("No audio files found. Check your download folder.")
else:
    start_time = time.time()
    processed_times = []

    # Transcribe each audio file
    for index, audio_file in enumerate(audio_files, 1):
        # Setup output filename for transcript
        transcript_title = extract_title(   # Use extract_title to get cleaned output filename for transcript
            audio_file.name,
            safe_filename=True
        )
        output_file = TRANSCRIPT_FOLDER / f"{transcript_title}.txt"

        # Skip files already transcribed !!!
        if output_file.exists():
            print(f"[{index}/{total_files}] Skipping: {audio_file.name}")
            continue
        print(f"\n[{index}/{total_files}] Transcribing: {audio_file.name}")
        file_start = time.time()
        
        try:
            # Use original audio path for transcription
            result = model.transcribe(
                str(audio_file),
                language="en",
                fp16=False,
                verbose=False,
            )
            # Save timestamped transcript
            with output_file.open("w", encoding="utf-8") as f:
                for segment in result["segments"]:
                    start = segment["start"]
                    text = segment["text"].strip()

                    minutes = int(start // 60)
                    seconds = int(start % 60)

                    f.write(f"[{minutes:02d}:{seconds:02d}] {text}\n")
            
            # Calculate processing time
            elapsed = time.time() - file_start
            processed_times.append(elapsed)
            # Estimate remaining time
            remaining_files = sum(
                1 for remaining_file in audio_files[index:]
                if not (
                    TRANSCRIPT_FOLDER /
                    f"{extract_title(remaining_file.name, safe_filename=True)}.txt"
                ).exists()
            )
            average_time = sum(processed_times) / len(processed_times)
            remaining_seconds = remaining_files * average_time
            minutes, seconds = divmod(int(remaining_seconds), 60)
            print(f"Saved: {output_file.name}")
            print(
                f"Progress: {index}/{total_files} | "
                f"Elapsed: {(time.time() - start_time)/60:.1f} min | "
                f"ETA: {minutes}m {seconds}s"
            )
        except Exception as error:
            print(f"ERROR processing {audio_file.name}: {error}")

    print("\nAll transcription jobs completed.") # Run on my GPUless laptop took 304.3 min

FFmpeg version: 7.1-essentials_build-www.gyan.dev

Found 130 recordings.
Loading Whisper model: tiny
[1/130] Skipping: 001 - Men in tech #YesAllMen.opus
[2/130] Skipping: 002 - Endocrine Biohacking, 2 years on (Everything is on Fire Edition).opus
[3/130] Skipping: 003 - PowerPoint Karaoke： After Dark Edition.opus
[4/130] Skipping: 004 - It’s the End of RSA as We Know It (And I Feel Fine)： An Introduction to Post-Quantum Cryptography.opus
[5/130] Skipping: 005 - EMF Variety Night.opus
[6/130] Skipping: 006 - Nerd Mentality： Duel! to the DEATH!.opus
[7/130] Skipping: 007 - Why aren't we making video games for pensioners？.opus
[8/130] Skipping: 008 - John Robertson's The Dark Room.opus
[9/130] Skipping: 009 - Expanded Radio Art.opus
[10/130] Skipping: 010 - Narrative Animals And How To Tell Better Stories.opus
[11/130] Skipping: 011 - Design decisions behind the Tildagon - a technical walkthrough.opus
[12/130] Skipping: 012 - Stimulating Brains Without Touching Them： A Story of Magnets, H

In [39]:
transcripts_df = pd.DataFrame(
    [
        {
            "title": transcript_file.stem,
            "transcript": transcript_file.read_text(encoding="utf-8"),
        }
        for transcript_file in sorted(TRANSCRIPT_FOLDER.glob("*.txt"))
    ],
    columns=["title", "transcript"],
)
#print(transcripts_df.loc[transcripts_df["title"] == "Expanded Radio Art", "transcript"].iloc[0])
#print(transcripts_df.loc[transcripts_df["title"] == "Expanded Radio Art", "transcript"].iloc[0][0:1000])
print(transcripts_df.head(2))
transcripts_df.to_csv("../../EMF_Camp/Raw_data/transcripts.csv", index=False, encoding="utf-8")

                                               title  \
0  3D Modelmaking ＂for girls＂ - The overlooked en...   
1  A 3 million dollar question： how fast does a m...   

                                          transcript  
0  [00:00] Hello.\n[00:17] Hello.\n[00:19] My nam...  
1  [00:00] I'm going to talk a bit about particle...  
